# Medical Discharge Summary → Indian Lay English Pipeline (v2 — Robust)

**Platform:** Kaggle Notebook (GPU T4 x2 or P100)

### Key improvements over v1
- **Sentence-level chunked processing** — splits 600-word documents into ~200-word chunks matching training data distribution
- **Entity-preserving lay dictionary** — outputs "high BP (hypertension)" format so medical terms are never lost
- **Quality-gated model generation** — strict hallucination detection; falls back to rule-based output when model fails
- **Post-processing entity verification** — re-injects any missing critical medical entities

## Before Running
1. **Add all datasets** via *Add Data → Your Datasets* (right panel):
   - `anotated_dataset_v2.xlsx`
   - `data/` folder (containing `data.tsv`)
   - `mtsamples.csv/`
   - `predictions-t5-base.tsv`
   - `PMC-Patients-V2.json`
2. **Internet:** Settings → Internet → **ON**
3. **Accelerator:** Settings → Accelerator → **GPU T4 x2**
4. After installation completes in Step 0, use **Kernel → Restart** then run all remaining cells.

## Step 0 — Local Library Installation (Kaggle-safe)

Installs all required packages into `/kaggle/working/local_libs/` so they take priority over Kaggle's pre-installed system packages. This avoids version conflicts (e.g., spaCy 3.9+ breaking scispaCy).

In [ ]:
import subprocess, sys, os

LOCAL_LIB = "/kaggle/working/local_libs"
os.makedirs(LOCAL_LIB, exist_ok=True)

if LOCAL_LIB not in sys.path:
    sys.path.insert(0, LOCAL_LIB)

def _pip_install(*args):
    """Install packages into the local lib directory."""
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "--target", LOCAL_LIB, *args],
        check=True
    )

print("Installing libraries into local_libs/ ...")
_pip_install("numpy==1.26.4")
_pip_install("pandas==2.2.2", "openpyxl")
_pip_install("spacy>=3.7.4,<3.8.0")
_pip_install("transformers", "sentencepiece", "accelerate", "packaging")
_pip_install("scispacy")
_pip_install(
    "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz"
)
_pip_install("textstat", "bert-score", "tqdm")

# ── IndicTrans2 dependencies (Step 12 — Regional Language Translation) ──
_pip_install("IndicTransToolkit")

print("  OK  All libraries installed to", LOCAL_LIB)
print("  --> Restart the kernel now, then run all remaining cells.")

## Step 1 — Imports

Load all required libraries after the kernel restart.

In [ ]:
import sys, os

LOCAL_LIB = "/kaggle/working/local_libs"
sys.path = [p for p in sys.path if p != LOCAL_LIB]
sys.path.insert(0, LOCAL_LIB)

import re
import glob
import warnings
import pandas as pd
import numpy as np
import spacy
import textstat
import torch
from tqdm import tqdm
from collections import Counter
from torch.utils.data import Dataset as TorchDataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import packaging.version as _pv
import transformers

warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_OFFLINE"] = "0"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"transformers {transformers.__version__} | torch {torch.__version__}")

## Step 2 — Load All Datasets

Auto-locates each file under `/kaggle/input/` using glob, loads all 7 data sources:

| Dataset | Rows | Role |
|---|---|---|
| `anotated_dataset_v2.xlsx` (annotated_dataset) | 200 | Primary input |
| `anotated_dataset_v2.xlsx` (abbreviations) | 301 | Abbreviation dictionary |
| `anotated_dataset_v2.xlsx` (ICD Codes) | ~2,000 | ICD code lookup |
| `data.tsv` | 93,368 | **Fine-tuning data** (original ↔ simplified pairs) |
| `predictions-t5-base.tsv` | 298 | Benchmarking |
| `mtsamples.csv` | 4,999 | Supplementary medical transcriptions |
| `PMC-Patients-V2.json` | 250,294 | PubMed patient case reports |

In [ ]:
def _locate(pattern):
    """Find a file under /kaggle/input/ or fall back to filename."""
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    return hits[0] if hits else pattern

# --- 2a. Annotated discharge summaries (200 rows) ---
ANNOTATED_PATH = _locate("anotated_dataset_v2.xlsx")
df = pd.read_excel(ANNOTATED_PATH, sheet_name="annotated_dataset")
print(f"[annotated_dataset] {len(df)} rows | cols: {list(df.columns)}")

# --- 2b. Abbreviations dictionary ---
abbr_df = pd.read_excel(ANNOTATED_PATH, sheet_name="abbreviations")
abbr_dict = dict(
    zip(
        abbr_df["abbreviated word"].astype(str).str.lower().str.strip(),
        abbr_df["translated"].astype(str).str.strip(),
    )
)
print(f"[abbreviations]     {len(abbr_dict)} entries")

# --- 2c. ICD Codes dictionary (auto-detect header row) ---
def _find_col(frame, *candidates):
    """Return first matching column name (case-insensitive)."""
    lower_map = {str(c).lower().strip(): c for c in frame.columns}
    for cand in candidates:
        hit = lower_map.get(cand.lower().strip())
        if hit:
            return hit
    return None

icd_df = None
for _hdr in range(5):
    _tmp = pd.read_excel(ANNOTATED_PATH, sheet_name="ICD Codes", header=_hdr)
    if not all(str(c).startswith("Unnamed") for c in _tmp.columns):
        icd_df = _tmp
        break
if icd_df is None:
    icd_df = pd.read_excel(ANNOTATED_PATH, sheet_name="ICD Codes", header=None)
    icd_df.columns = [f"col_{i}" for i in range(icd_df.shape[1])]

_k = _find_col(icd_df, "CID Subcategory", "ICD Subcategory", "Subcategory", "Code", "col_0")
_v = _find_col(icd_df, "Translated Subcat Description", "Translated Description", "Description", "col_1")
icd_dict = dict(zip(
    icd_df[_k].astype(str).str.lower().str.strip(),
    icd_df[_v].astype(str).str.strip()
)) if _k and _v else {}
print(f"[ICD Codes]         {len(icd_dict)} entries  (key='{_k}', val='{_v}')")

# --- 2d. data.tsv — fine-tuning data (original <-> simplified pairs) ---
DATA_TSV_PATH = _locate("data.tsv")
finetune_df = pd.read_csv(DATA_TSV_PATH, sep="\t", low_memory=False)
print(f"[data.tsv]          {len(finetune_df)} rows | cols: {list(finetune_df.columns)}")

# --- 2e. predictions-t5-base.tsv — for benchmarking ---
T5_PRED_PATH = _locate("predictions-t5-base.tsv")
t5_df = pd.read_csv(T5_PRED_PATH, sep="\t")
print(f"[predictions-t5]    {len(t5_df)} rows | cols: {list(t5_df.columns)}")

# --- 2f. mtsamples.csv — additional medical transcriptions ---
MT_PATH = _locate("mtsamples.csv")
mt_df = pd.read_csv(MT_PATH)
print(f"[mtsamples.csv]     {len(mt_df)} rows | cols: {list(mt_df.columns)}")

# --- 2g. PMC-Patients-V2.json ---
PMC_PATH = _locate("PMC-Patients-V2.json")
print(f"[PMC-Patients]      located: {PMC_PATH}")

## Step 3 — Data Analytics (Why v1 failed)

Before building the pipeline, we analyze the data to understand why v1 produced hallucinated outputs with lost medical entities.

In [ ]:
# ── Length distribution: training data vs discharge summaries ──────
ft_sample = finetune_df[["original", "english simplified"]].dropna().head(10000)
ft_orig_len = ft_sample["original"].str.split().str.len()
ft_simp_len = ft_sample["english simplified"].str.split().str.len()
ds_len = df["translated discharge_summary"].dropna().str.split().str.len()

print("=== Length Analysis (words) ===")
print(f"  data.tsv originals :  mean={ft_orig_len.mean():.0f}  median={ft_orig_len.median():.0f}  max={ft_orig_len.max():.0f}")
print(f"  data.tsv simplified:  mean={ft_simp_len.mean():.0f}  median={ft_simp_len.median():.0f}  max={ft_simp_len.max():.0f}")
print(f"  Discharge summaries:  mean={ds_len.mean():.0f}  median={ds_len.median():.0f}  max={ds_len.max():.0f}")
print()
print("  PROBLEM: Discharge summaries are ~2.6x longer than training data.")
print("  SOLUTION: Split into ~200-word chunks before model inference.")
print()

# ── Domain check ──────────────────────────────────────────────────
print("=== Domain Check ===")
print("  data.tsv sample topics: PubMed research abstracts (facelift surgery, prostate cancer research, etc.)")
print("  Discharge summaries:    Clinical patient records (diagnoses, treatments, lab values)")
print("  PROBLEM: Domain mismatch — model learns research language, not clinical language.")
print("  SOLUTION: Fine-tune + use quality gates to reject hallucinated output; rule-based fallback.")
print()

# ── Entity frequency in discharge summaries ───────────────────────
key_terms = ["sepsis", "pneumonia", "diabetes", "hypertension", "kidney",
             "anemia", "tuberculosis", "amputation", "infarction", "epileptic"]
print("=== Top medical terms in discharge summaries ===")
for term in sorted(key_terms, key=lambda t: df["translated discharge_summary"].str.contains(t, case=False, na=False).sum(), reverse=True):
    count = df["translated discharge_summary"].str.contains(term, case=False, na=False).sum()
    print(f"  {term:20s}  appears in {count:3d}/200 summaries")
print()
print("  These entities MUST be preserved in the simplified output.")

## Step 4 — Preprocess, Sentence Split, NER & Abbreviation Expansion

For each discharge summary:
1. **Clean** placeholders (`[person]` → `the patient`, `{organization}` → `the hospital`)
2. **Expand abbreviations** using 301-entry dictionary
3. **Split into sentences** using spaCy
4. **Extract medical entities** per sentence using scispaCy `en_core_sci_md`
5. **Group into ~200-word chunks** matching training data length distribution

In [ ]:
# ── Load scispaCy model (sentence splitting + NER) ────────────────
print("Loading scispaCy en_core_sci_md ...")
nlp = spacy.load("en_core_sci_md")
nlp.max_length = 2_000_000
print("  OK  scispaCy model loaded.")

# ── Helper functions ──────────────────────────────────────────────
def preprocess_text(text):
    """Collapse whitespace, strip."""
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip()

def clean_placeholders(text):
    """Replace [person], {organization}, etc."""
    text = re.sub(r"\[person\]", "the patient", text, flags=re.IGNORECASE)
    text = re.sub(r"\{organization\}", "the hospital", text, flags=re.IGNORECASE)
    text = re.sub(r"\{city\}", "the city", text, flags=re.IGNORECASE)
    text = re.sub(r"\{omitted\}", "", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()

def expand_abbreviations(text, abbr_dict):
    """Replace abbreviated tokens with full forms (punctuation-aware)."""
    if not isinstance(text, str):
        return ""
    tokens = text.split()
    expanded = []
    for token in tokens:
        stripped = token.strip(".,;:!?()[]{}\"'")
        lookup = stripped.lower()
        if lookup in abbr_dict:
            idx_s = token.find(stripped)
            prefix = token[:idx_s]
            suffix = token[idx_s + len(stripped):]
            expanded.append(prefix + abbr_dict[lookup] + suffix)
        else:
            expanded.append(token)
    return " ".join(expanded)

def sentence_split(text):
    """Split text into sentences using spaCy."""
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

def extract_entities(text):
    """Extract medical entities from text."""
    if not isinstance(text, str) or not text.strip():
        return []
    doc = nlp(text)
    return [{"entity": e.text, "label": e.label_} for e in doc.ents]

def chunk_sentences(sent_data_list, max_words=200):
    """Group sentences into chunks of ~max_words to match training data length."""
    chunks = []
    current_chunk = []
    current_words = 0
    for sd in sent_data_list:
        sw = len(sd["text"].split())
        if current_words + sw > max_words and current_chunk:
            chunks.append(current_chunk)
            current_chunk = [sd]
            current_words = sw
        else:
            current_chunk.append(sd)
            current_words += sw
    if current_chunk:
        chunks.append(current_chunk)
    return chunks

# ── Process all summaries ─────────────────────────────────────────
print("Processing 200 discharge summaries ...")
all_processed = []

for idx in tqdm(range(len(df)), desc="Step 4"):
    raw = str(df.at[idx, "translated discharge_summary"])

    # Clean + expand abbreviations
    text = preprocess_text(raw)
    text = clean_placeholders(text)
    text = expand_abbreviations(text, abbr_dict)

    # Split into sentences + extract entities per sentence
    sentences = sentence_split(text)
    sent_data = []
    all_ents = []
    for sent in sentences:
        ents = extract_entities(sent)
        all_ents.extend(ents)
        sent_data.append({
            "text": sent,
            "entities": ents,
            "entity_texts": set(e["entity"].lower() for e in ents),
        })

    # Group into ~200-word chunks
    chunks = chunk_sentences(sent_data, max_words=200)

    all_processed.append({
        "idx": idx,
        "original": raw,
        "expanded": text,
        "sentence_data": sent_data,
        "chunks": chunks,
        "all_entities": all_ents,
    })

df["expanded_summary"] = [p["expanded"] for p in all_processed]
avg_sents = np.mean([len(p["sentence_data"]) for p in all_processed])
avg_chunks = np.mean([len(p["chunks"]) for p in all_processed])
avg_ents = np.mean([len(p["all_entities"]) for p in all_processed])
print(f"  OK  Avg {avg_sents:.1f} sentences, {avg_chunks:.1f} chunks, {avg_ents:.1f} entities per summary")

## Step 5 — Entity-Preserving Lay Dictionary + Rule-Based Simplification

The **Indian Lay Dictionary** replaces medical terms with simple equivalents common in Indian English.  
**Key change from v1:** For recognized NER entities, the output preserves **both** forms:

| Medical Term | Entity-Preserving Output |
|---|---|
| hypertension | high BP (hypertension) |
| myocardial infarction | heart attack (myocardial infarction) |
| pneumonia | lung infection (pneumonia) |
| sepsis | serious blood infection (sepsis) |
| diabetes mellitus | sugar disease (diabetes) |

Non-entity terms (e.g., "administer" → "give") are replaced directly.

The **rule-based output** is always generated and serves as a reliable fallback when the model hallucinates.

In [ ]:
INDIAN_LAY_DICT = {
    # ── Conditions ────────────────────────────────────────────────
    "hypertension": "high BP",
    "diabetes mellitus": "sugar disease (diabetes)",
    "myocardial infarction": "heart attack",
    "acute renal failure": "sudden kidney failure",
    "chronic renal failure": "long-term kidney problem",
    "renal failure": "kidney failure",
    "kidney disease": "kidney problem",
    "septicemia": "blood infection (sepsis)",
    "sepsis": "serious blood infection",
    "septic shock": "severe infection causing dangerously low BP",
    "pneumonia": "lung infection",
    "aspiration pneumonia": "lung infection from inhaling food/liquid",
    "tuberculosis": "TB (tuberculosis)",
    "pneumothorax": "air trapped in chest",
    "endocarditis": "infection of heart valves",
    "spondylodiscitis": "spine bone infection",
    "hypotension": "low BP",
    "pyrexia": "fever",
    "dyspnea": "difficulty in breathing",
    "edema": "swelling",
    "abscess": "pus-filled swelling",
    "crohn's disease": "long-term bowel disease (Crohn's)",
    "cerebrovascular accident": "brain stroke",
    "anemia": "low blood (anemia)",
    "tachycardia": "fast heartbeat",
    "bradycardia": "slow heartbeat",
    "atrial fibrillation": "irregular heartbeat",
    "cardiac arrest": "heart stopped beating",
    "respiratory failure": "lungs unable to work properly",
    "urinary tract infection": "urine infection (UTI)",
    "encephalopathy": "brain dysfunction",
    "ischemic": "reduced blood supply",
    "hypoxic": "low oxygen",
    "necrosis": "tissue death",
    "thrombosis": "blood clot",
    "embolism": "blood clot blocking a vessel",
    "peritonitis": "infection inside the abdomen",
    "cellulitis": "skin infection",
    "osteomyelitis": "bone infection",
    "meningitis": "brain membrane infection",
    "cirrhosis": "liver scarring",
    "ascites": "fluid buildup in abdomen",
    "pleural effusion": "fluid around the lungs",
    "status epilepticus": "prolonged seizure",
    "epilepsy": "seizure disorder",
    "hypoglycemia": "very low blood sugar",
    "hyperglycemia": "very high blood sugar",
    # ── Procedures & Treatments ───────────────────────────────────
    "hemodialysis": "kidney dialysis",
    "tracheostomy": "breathing tube in neck",
    "thoracotomy": "chest surgery",
    "pneumonectomy": "removal of a lung",
    "ileocolectomy": "removal of part of bowel",
    "anastomosis": "surgical joining of bowel ends",
    "debridement": "surgical wound cleaning",
    "arteriovenous fistula": "dialysis access point on arm",
    "intubation": "breathing tube placed in throat",
    "extubation": "breathing tube removed",
    "mechanical ventilation": "machine-assisted breathing",
    "vasopressors": "medicines to raise blood pressure",
    "intramuscular": "given as a muscle injection",
    "intravenous": "given through a vein (drip/IV)",
    "percutaneous": "through the skin",
    "antibiotic therapy": "antibiotic treatment",
    "antibiotic": "infection-fighting medicine",
    "computed tomography": "CT scan",
    "magnetic resonance": "MRI scan",
    "echocardiogram": "heart ultrasound",
    "electrocardiogram": "ECG/heart tracing",
    # ── Lab & Medical Terms ───────────────────────────────────────
    "blood culture": "blood test to find infection",
    "procalcitonin": "blood marker for infection",
    "hemoglobin": "blood count (Hb)",
    "creatinine": "kidney function marker",
    "bilirubin": "liver function marker",
    "saturation": "oxygen level in blood",
    "leukocytosis": "high white blood cell count",
    "thrombocytopenia": "low platelet count",
    "afebrile": "no fever",
    "eupneic": "breathing normally",
    "acyanotic": "no bluish discoloration",
    "anicteric": "no yellowing of skin/eyes",
    "vesicular breath sounds": "normal breathing sounds",
    "hemodynamically stable": "stable blood pressure and heart rate",
    "level of consciousness": "alertness level",
    # ── Hospital / Admin ──────────────────────────────────────────
    "discharge": "sent home from hospital",
    "outpatient clinic": "OPD (Out Patient Department)",
    "follow-up": "return visit to doctor",
    "ward": "general hospital room",
    "intensive care unit": "ICU (serious care room)",
    "sus": "government health scheme",
    "orally": "by mouth",
    "administer": "give",
    "prognosis": "expected outcome",
    "etiology": "cause of disease",
    "comorbidity": "other existing health condition",
    "nosocomial": "hospital-acquired",
    "palliative": "comfort-focused (not curative)",
}

def apply_entity_preserving_lay_dict(text, lay_dict, entity_texts):
    """Replace medical terms with lay equivalents.
    For NER entities: 'lay_term (medical_term)' to preserve both.
    For non-entities: straight replacement."""
    result = text.lower()
    for key in sorted(lay_dict, key=len, reverse=True):
        if key not in result:
            continue
        lay_term = lay_dict[key]
        # Is this term (or part of it) a recognized entity?
        is_entity = any(key in ent or ent in key for ent in entity_texts)
        if is_entity:
            # Lay term already has parens (e.g., "sugar disease (diabetes)")? Don't double up.
            if "(" in lay_term:
                replacement = lay_term
            else:
                replacement = f"{lay_term} ({key})"
        else:
            replacement = lay_term
        result = result.replace(key, replacement)
    return result

def capitalize_sentences(text):
    """Capitalize first letter of each sentence."""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return " ".join(s[0].upper() + s[1:] if s else s for s in sentences)

# ── Apply rule-based simplification to all chunks ─────────────────
print("Applying entity-preserving lay dictionary ...")
for proc in tqdm(all_processed, desc="Step 5"):
    rule_sentences = []
    for sd in proc["sentence_data"]:
        simplified = apply_entity_preserving_lay_dict(
            sd["text"], INDIAN_LAY_DICT, sd["entity_texts"]
        )
        simplified = capitalize_sentences(simplified)
        sd["rule_based"] = simplified
        rule_sentences.append(simplified)

    # Also build rule-based per-chunk (for fallback)
    for chunk in proc["chunks"]:
        chunk_rule = " ".join(sd["rule_based"] for sd in chunk)
        for sd in chunk:
            sd["chunk_rule"] = chunk_rule

    proc["rule_based_full"] = " ".join(rule_sentences)

df["rule_based_simplified"] = [p["rule_based_full"] for p in all_processed]
print("  OK  Rule-based simplification complete.")
print()

# Show before/after for first summary
print("=== Example: Row 0 ===")
print("ORIGINAL (first 300 chars):")
print(all_processed[0]["original"][:300])
print()
print("RULE-BASED (first 300 chars):")
print(all_processed[0]["rule_based_full"][:300])

## Step 6 — Fine-Tune IndicBART (Sentence-Level)

Fine-tunes `ai4bharat/IndicBART` on **10,000 pairs** from `data.tsv`
(`original` → `english simplified`) using a **PyTorch Dataset** (no HuggingFace `datasets` library).

| Parameter | Value |
|---|---|
| Base model | `ai4bharat/IndicBART` (~244 M params) |
| Training rows | 10,000 (from 93 K in data.tsv) |
| Train / Eval split | 90 % / 10 % |
| Epochs | 3 |
| Batch size | 4 per device |
| Learning rate | 3 e-5 |
| Precision | FP16 (GPU) |
| Output | `/kaggle/working/indicbart-finetuned/` |

> Uses `processing_class` instead of `tokenizer` for transformers ≥ 4.46.

In [ ]:
MODEL_NAME      = "ai4bharat/IndicBART"
FINETUNE_ROWS   = 10000
FINETUNE_EPOCHS = 3
FINETUNE_BATCH  = 2       # lowered from 4 — prevents OOM on 15 GB GPU
FINETUNE_LR     = 3e-5
MAX_SRC_LEN     = 256
MAX_TGT_LEN     = 256
OUTPUT_DIR      = "/kaggle/working/indicbart-finetuned"

# ── Free GPU memory from earlier steps before loading model ────────
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f"GPU memory free: {free_gb:.2f} GB")

# ── Filter valid fine-tuning rows ─────────────────────────────────
ft = finetune_df[["original", "english simplified"]].dropna()
ft = ft[
    ft["original"].str.strip().astype(bool) &
    ft["english simplified"].str.strip().astype(bool)
]
ft = ft.head(FINETUNE_ROWS).reset_index(drop=True)
print(f"Fine-tuning samples: {len(ft)}")
print(f"  Avg original words : {ft['original'].str.split().str.len().mean():.0f}")
print(f"  Avg simplified words: {ft['english simplified'].str.split().str.len().mean():.0f}")

# ── Load base model & tokenizer (CPU first, then move to GPU) ──────
print("\nLoading IndicBART base model ...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, do_lower_case=False, use_fast=False, keep_accents=True
)
# Load on CPU to avoid double-allocation when Trainer moves it
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, low_cpu_mem_usage=True)
base_model.config.tie_word_embeddings = False
# Enable gradient checkpointing: trades speed for ~30% less VRAM
base_model.gradient_checkpointing_enable()
DECODER_START_ID = tokenizer.convert_tokens_to_ids("<2en>")
print(f"  Decoder start token: <2en> → id {DECODER_START_ID}")

# ── PyTorch Dataset (replaces HuggingFace datasets library) ───────
class SimplificationDataset(TorchDataset):
    """Tokenized seq2seq dataset for medical text simplification."""
    def __init__(self, sources, targets, tok, max_src=256, max_tgt=256):
        self.tok = tok
        self.sources = ["<s> " + s + " </s> <2en>" for s in sources]
        self.targets = list(targets)
        self.max_src = max_src
        self.max_tgt = max_tgt

    def __len__(self):
        return len(self.sources)

    def __getitem__(self, idx):
        src = self.tok(
            self.sources[idx], max_length=self.max_src,
            truncation=True, padding="max_length", return_tensors="pt",
        )
        tgt = self.tok(
            self.targets[idx], max_length=self.max_tgt,
            truncation=True, padding="max_length", return_tensors="pt",
        )
        labels = tgt["input_ids"].squeeze()
        labels[labels == self.tok.pad_token_id] = -100
        return {
            "input_ids":      src["input_ids"].squeeze(),
            "attention_mask": src["attention_mask"].squeeze(),
            "labels":         labels,
        }

# ── Train / eval split (90/10) ────────────────────────────────────
n_eval = max(1, int(len(ft) * 0.1))
train_slice = ft.iloc[:-n_eval]
eval_slice  = ft.iloc[-n_eval:]

train_ds = SimplificationDataset(
    train_slice["original"].tolist(),
    train_slice["english simplified"].tolist(),
    tokenizer, MAX_SRC_LEN, MAX_TGT_LEN,
)
eval_ds = SimplificationDataset(
    eval_slice["original"].tolist(),
    eval_slice["english simplified"].tolist(),
    tokenizer, MAX_SRC_LEN, MAX_TGT_LEN,
)
print(f"  Train: {len(train_ds)} | Eval: {len(eval_ds)}")

# ── Seq2Seq training ──────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=FINETUNE_EPOCHS,
    per_device_train_batch_size=FINETUNE_BATCH,
    per_device_eval_batch_size=FINETUNE_BATCH,
    gradient_accumulation_steps=2,   # effective batch = 2×2 = 4
    learning_rate=FINETUNE_LR,
    weight_decay=0.01,
    warmup_steps=200,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    fp16=(device == "cuda"),
    logging_steps=100,
    report_to="none",
    load_best_model_at_end=True,
    generation_max_length=MAX_TGT_LEN,
    dataloader_pin_memory=False,    # reduces peak memory
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=base_model, padding=True)

# ── Use processing_class for transformers >= 4.46 ─────────────────
_trainer_kwargs = dict(
    model=base_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
)
if _pv.parse(transformers.__version__) >= _pv.parse("4.46.0"):
    _trainer_kwargs["processing_class"] = tokenizer
else:
    _trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**_trainer_kwargs)

print(f"\nTraining: {FINETUNE_EPOCHS} epochs | batch={FINETUNE_BATCH} | "
      f"grad_accum=2 | effective_batch=4 | lr={FINETUNE_LR}")
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"  OK  Fine-tuned model saved to {OUTPUT_DIR}")

## Step 7 — Chunk-Level Generation with Quality Gates

Generates simplified text **chunk by chunk** (~200 words each — matching training data distribution).

For **each chunk**:
1. Run fine-tuned IndicBART → `model_output`
2. **Strip** decoder prefix (`<2en>`, `<s>`, etc.)
3. **Quality checks** (ALL must pass or the chunk falls back to rule-based):
   - No hallucination markers (`"this study"`, `"researchers found"`, etc.)  
   - Entity recall ≥ 40 % (medical terms from Step 4 must survive)
   - Length ratio 0.25–3.0 (output not absurdly short/long vs input)
4. **If passed** → apply lay dictionary on model output (ensures Indian terms)
5. **If failed** → use the rule-based version from Step 5

Checkpoints are saved every 50 summaries to avoid losing progress.

In [ ]:
# ── Load fine-tuned model ──────────────────────────────────────────
model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR).to(device)
model.eval()
print(f"Fine-tuned model loaded on {device}")

# ── Generation helper ─────────────────────────────────────────────
def generate_chunk(text, model, tokenizer, max_input=512, max_output=300):
    """Generate simplified text for a single chunk."""
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""
    try:
        input_text = "<s> " + text + " </s> <2en>"
        inputs = tokenizer(
            input_text, max_length=max_input, truncation=True,
            padding=True, return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_length=max_output,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=3,
                decoder_start_token_id=DECODER_START_ID,
            )
        return tokenizer.decode(output_ids[0], skip_special_tokens=True)
    except Exception as e:
        print(f"  Warning: generation failed — {e}")
        return ""

# ── Post-processing: strip decoder artefacts ──────────────────────
def clean_model_output(text):
    """Remove <2en>, <s>, </s> prefixes and clean whitespace."""
    text = re.sub(r"^(<2?\w+>|</?s>)\s*", "", text).strip()
    text = re.sub(r"^(<2?\w+>|</?s>)\s*", "", text).strip()  # second pass
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ── Quality gate ──────────────────────────────────────────────────
HALLUCINATION_MARKERS = [
    "this study", "this research", "researchers found",
    "the study found", "our research", "in this paper",
    "the present study", "the authors found",
]

def passes_quality_check(original, model_output, entity_texts):
    """Return True if model output is acceptable, False → fall back to rule-based."""
    if not model_output or len(model_output.strip()) < 10:
        return False

    orig_lower = original.lower()
    out_lower  = model_output.lower()

    # Reject hallucination markers absent from original
    for marker in HALLUCINATION_MARKERS:
        if marker in out_lower and marker not in orig_lower:
            return False

    # Entity recall: at least 40 % of entities must appear in output
    if entity_texts:
        preserved = sum(
            1 for ent in entity_texts
            if ent in out_lower or any(
                w in out_lower for w in ent.split() if len(w) > 3
            )
        )
        recall = preserved / len(entity_texts)
        if recall < 0.4:
            return False

    # Length ratio: reject absurdly short or long output
    orig_wc = len(original.split())
    out_wc  = len(model_output.split())
    if orig_wc > 5:
        ratio = out_wc / orig_wc
        if ratio < 0.25 or ratio > 3.0:
            return False

    return True

# ── Main generation loop (chunk-level) ────────────────────────────
CHECKPOINT_INTERVAL = 50
model_used_count    = 0
rule_fallback_count = 0

print(f"\nGenerating simplified text for {len(all_processed)} summaries ...")
for proc in tqdm(all_processed, desc="Step 7"):
    final_chunks = []

    for chunk in proc["chunks"]:
        # Build chunk input text and collect entities
        chunk_text = " ".join(sd["text"] for sd in chunk)
        chunk_entities = set()
        for sd in chunk:
            chunk_entities.update(sd["entity_texts"])

        # Generate with model
        raw_output = generate_chunk(chunk_text, model, tokenizer)
        cleaned    = clean_model_output(raw_output)

        if passes_quality_check(chunk_text, cleaned, chunk_entities):
            # Model passed — apply lay dict on model output
            final = apply_entity_preserving_lay_dict(
                cleaned, INDIAN_LAY_DICT, chunk_entities
            )
            final = capitalize_sentences(final)
            model_used_count += 1
        else:
            # Fallback to rule-based
            final = " ".join(sd["rule_based"] for sd in chunk)
            rule_fallback_count += 1

        final_chunks.append(final)

    proc["final_simplified"] = " ".join(final_chunks)

    # Checkpoint
    idx = proc["idx"]
    if (idx + 1) % CHECKPOINT_INTERVAL == 0 or idx == len(all_processed) - 1:
        df.loc[:idx, "indian_lay_english"] = [
            all_processed[i]["final_simplified"] for i in range(idx + 1)
        ]
        df.to_excel(
            f"/kaggle/working/checkpoint_{idx+1}.xlsx", index=False
        )
        print(f"  Checkpoint saved at row {idx+1}")

# Final write
df["indian_lay_english"] = [p["final_simplified"] for p in all_processed]

total_chunks = model_used_count + rule_fallback_count
print(f"\n  OK  Generation complete.")
print(f"  Model accepted : {model_used_count}/{total_chunks} chunks "
      f"({100*model_used_count/max(total_chunks,1):.1f}%)")
print(f"  Rule fallback  : {rule_fallback_count}/{total_chunks} chunks "
      f"({100*rule_fallback_count/max(total_chunks,1):.1f}%)")

df[["translated discharge_summary", "indian_lay_english"]].head(2)

## Step 8 — Entity Verification & Final Assembly

Post-processing safety net:

1. Compare medical entities extracted in Step 4 against the final simplified text.
2. For each **missing critical entity**, re-inject it as a bracketed note:  
   `[Other medical details: serious blood infection (sepsis), low blood (anemia).]`
3. Store final output in `df["indian_lay_english"]` and entity audit in `df["medical_entities"]`.

In [ ]:
def verify_and_inject_entities(text, entities, lay_dict):
    """Verify all critical entities are present. Re-inject missing ones."""
    text_lower = text.lower()
    missing = []
    for ent in entities:
        ent_text = ent["entity"].lower()
        if len(ent_text) <= 2:            # skip trivial tokens
            continue
        lay_equiv = lay_dict.get(ent_text, "")
        # Check if entity OR its lay equivalent is present
        if (ent_text not in text_lower
                and lay_equiv.lower() not in text_lower
                and not any(w in text_lower for w in ent_text.split() if len(w) > 3)):
            missing.append(ent_text)

    # Deduplicate preserving order
    unique_missing = list(dict.fromkeys(missing))

    if unique_missing and len(unique_missing) <= 15:
        notes = []
        for ent in unique_missing:
            if ent in lay_dict:
                notes.append(f"{lay_dict[ent]} ({ent})")
            else:
                notes.append(ent)
        text += " [Other medical details: " + ", ".join(notes) + ".]"
    return text

# ── Apply entity verification ─────────────────────────────────────
print("Running entity verification on all 200 summaries ...")
entity_gaps = []  # track how many missing per summary
for proc in tqdm(all_processed, desc="Step 8"):
    original_text = proc["final_simplified"]
    entities      = proc["all_entities"]
    verified      = verify_and_inject_entities(original_text, entities, INDIAN_LAY_DICT)
    proc["verified_output"] = verified

    # Count what was missing
    missing_count = len(verified) - len(original_text)
    entity_gaps.append(1 if missing_count > 0 else 0)

df["indian_lay_english"] = [p["verified_output"] for p in all_processed]
df["medical_entities"]   = [str(p["all_entities"]) for p in all_processed]

gap_total = sum(entity_gaps)
print(f"  OK  Entity verification complete.")
print(f"  Summaries needing entity injection: {gap_total}/{len(all_processed)}")
print()

# ── Quick entity recall audit ─────────────────────────────────────
key_terms = ["sepsis", "pneumonia", "hypertension", "diabetes", "anemia",
             "kidney", "tuberculosis", "epilep"]
print("=== Entity Recall Audit ===")
for term in key_terms:
    in_orig = df["translated discharge_summary"].str.contains(term, case=False, na=False).sum()
    in_out  = df["indian_lay_english"].str.contains(term, case=False, na=False).sum()
    lay_check = [v for k, v in INDIAN_LAY_DICT.items() if term in k]
    in_lay  = sum(
        df["indian_lay_english"].str.contains(re.escape(lv), case=False, na=False).sum()
        for lv in lay_check
    ) if lay_check else 0
    print(f"  {term:20s}  original={in_orig:3d}  output={in_out:3d}  lay_equiv={in_lay:3d}")
print()
df[["translated discharge_summary", "indian_lay_english"]].head(2)

## Step 9 — Evaluation

Computes **Flesch Reading Ease**, word counts, and compression ratio for each summary pair.  
Also compares the **rule-based** and **final** (model+rule hybrid) outputs.

| Flesch Score | Readability |
|---|---|
| 90–100 | Very Easy (5th grade) |
| 60–70 | Standard (8th–9th grade) |
| 30–50 | Difficult (college level) |
| 0–30 | Very Difficult (medical/legal) |

In [ ]:
def evaluate_summary(original, simplified):
    """Compute Flesch readability, word counts, compression ratio."""
    if not isinstance(original, str) or not original.strip():
        original = ""
    if not isinstance(simplified, str) or not simplified.strip():
        simplified = ""
    flesch_orig = textstat.flesch_reading_ease(original) if original else 0.0
    flesch_simp = textstat.flesch_reading_ease(simplified) if simplified else 0.0
    wc_orig = len(original.split())
    wc_simp = len(simplified.split())
    return {
        "flesch_original":    flesch_orig,
        "flesch_simplified":  flesch_simp,
        "flesch_improvement": flesch_simp - flesch_orig,
        "word_count_original":    wc_orig,
        "word_count_simplified":  wc_simp,
        "compression_ratio": wc_simp / wc_orig if wc_orig > 0 else 0.0,
    }

print("Running evaluation ...")

# Evaluate final output
eval_results = [
    evaluate_summary(
        df.at[i, "translated discharge_summary"],
        df.at[i, "indian_lay_english"],
    )
    for i in tqdm(range(len(df)), desc="Eval final")
]
eval_df = pd.DataFrame(eval_results)
for col in eval_df.columns:
    df[col] = eval_df[col].values

# Also evaluate rule-based output for comparison
rule_eval = [
    evaluate_summary(
        df.at[i, "translated discharge_summary"],
        df.at[i, "rule_based_simplified"],
    )
    for i in range(len(df))
]
rule_eval_df = pd.DataFrame(rule_eval)
df["flesch_rule_based"] = rule_eval_df["flesch_simplified"].values

print("  OK  Evaluation complete.\n")
print("=== Final Output ===")
print(eval_df.describe().round(2).to_string())
print(f"\n=== Rule-Based Comparison ===")
print(f"  Avg Flesch (rule-based): {df['flesch_rule_based'].mean():.2f}")
print(f"  Avg Flesch (final)     : {df['flesch_simplified'].mean():.2f}")

## Step 10 — Save Output (Formatted Excel + HTML Report)

Saves results to **two files** for easy reading:

| File | Best for |
|---|---|
| `lay_english_pipeline_output.xlsx` | Excel — text wrapped, wide columns, alternating row colours |
| `lay_english_pipeline_output.html` | **Browser** — side-by-side original vs simplified, full-width readable paragraphs |

In [ ]:
import openpyxl
from openpyxl.styles import PatternFill, Alignment, Font
from openpyxl.utils import get_column_letter

OUTPUT_COLUMNS = [
    "translated discharge_summary",
    "expanded_summary",
    "rule_based_simplified",
    "indian_lay_english",
    "medical_entities",
    "translated diagnosis_ICD", "translanted outcome",
    "lenght of stay", "medical specialties",
    "flesch_original", "flesch_simplified", "flesch_improvement",
    "flesch_rule_based",
    "word_count_original", "word_count_simplified", "compression_ratio",
]
existing_cols = [c for c in OUTPUT_COLUMNS if c in df.columns]
missing_cols  = [c for c in OUTPUT_COLUMNS if c not in df.columns]
if missing_cols:
    print(f"  Warning: Missing columns (skipped): {missing_cols}")

OUTPUT_PATH = "/kaggle/working/lay_english_pipeline_output.xlsx"
HTML_PATH   = "/kaggle/working/lay_english_pipeline_output.html"

# ── 1. Write raw Excel ────────────────────────────────────────────
df[existing_cols].to_excel(OUTPUT_PATH, index=False, engine="openpyxl")

# ── 2. Re-open with openpyxl and apply formatting ─────────────────
wb = openpyxl.load_workbook(OUTPUT_PATH)
ws = wb.active

LONG_TEXT_COLS = {
    "translated discharge_summary",
    "expanded_summary",
    "rule_based_simplified",
    "indian_lay_english",
    "medical_entities",
}
NARROW_COLS = {
    "flesch_original", "flesch_simplified", "flesch_improvement",
    "flesch_rule_based",
    "word_count_original", "word_count_simplified", "compression_ratio",
    "lenght of stay",
}

HEADER_FILL  = PatternFill("solid", fgColor="1F4E79")
HEADER_FONT  = Font(bold=True, color="FFFFFF", size=11)
ROW_ODD      = PatternFill("solid", fgColor="EBF3FB")
ROW_EVN      = PatternFill("solid", fgColor="FFFFFF")
WRAP_TOP     = Alignment(wrap_text=True, vertical="top")
CENTER_MID   = Alignment(wrap_text=False, vertical="center", horizontal="center")

col_names = [ws.cell(1, c).value for c in range(1, ws.max_column + 1)]

for col_idx, col_name in enumerate(col_names, start=1):
    letter = get_column_letter(col_idx)
    hdr = ws.cell(1, col_idx)
    hdr.fill = HEADER_FILL
    hdr.font = HEADER_FONT
    hdr.alignment = CENTER_MID
    if col_name in LONG_TEXT_COLS:
        ws.column_dimensions[letter].width = 65
    elif col_name in NARROW_COLS:
        ws.column_dimensions[letter].width = 16
    else:
        ws.column_dimensions[letter].width = 24

for row_idx in range(2, ws.max_row + 1):
    fill = ROW_ODD if row_idx % 2 == 0 else ROW_EVN
    for col_idx, col_name in enumerate(col_names, start=1):
        cell = ws.cell(row_idx, col_idx)
        cell.fill = fill
        cell.alignment = WRAP_TOP if col_name in LONG_TEXT_COLS else Alignment(vertical="top")
    ws.row_dimensions[row_idx].height = 130   # ~8-10 lines visible

ws.freeze_panes = "A2"
wb.save(OUTPUT_PATH)
print(f"  OK  Formatted Excel  → {OUTPUT_PATH}")

# ── 3. Generate HTML report ───────────────────────────────────────
def _esc(s):
    return str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

rows_html = []
for i in range(len(df)):
    orig    = _esc(df.at[i, "translated discharge_summary"])
    rule_b  = _esc(df.at[i, "rule_based_simplified"]) if "rule_based_simplified" in df.columns else ""
    final   = _esc(df.at[i, "indian_lay_english"])
    flesch  = f"{df.at[i, 'flesch_simplified']:.1f}" if "flesch_simplified" in df.columns else ""
    crat    = f"{df.at[i, 'compression_ratio']:.2f}" if "compression_ratio" in df.columns else ""
    bg      = "#f0f7ff" if i % 2 == 0 else "#ffffff"
    rows_html.append(f"""
    <tr style='background:{bg}'>
      <td class='num'>{i+1}</td>
      <td class='text'>{orig}</td>
      <td class='text simp'>{rule_b}</td>
      <td class='text final'>{final}</td>
      <td class='num'>{flesch}</td>
      <td class='num'>{crat}</td>
    </tr>""")

html = f"""<!DOCTYPE html>
<html lang='en'>
<head>
<meta charset='UTF-8'>
<title>Medical Pipeline v2 — Results</title>
<style>
  body  {{ font-family:'Segoe UI',Arial,sans-serif; font-size:13px;
           margin:0; padding:16px; background:#f0f4f8; }}
  h1    {{ color:#1F4E79; margin-bottom:4px; }}
  p     {{ color:#555; margin-top:0; }}
  table {{ border-collapse:collapse; width:100%; background:#fff;
           box-shadow:0 1px 6px rgba(0,0,0,.12); border-radius:6px;
           overflow:hidden; }}
  th    {{ background:#1F4E79; color:#fff; padding:10px 14px;
           text-align:left; position:sticky; top:0; z-index:2;
           white-space:nowrap; }}
  td    {{ padding:10px 14px; vertical-align:top;
           border-bottom:1px solid #dde6f0; }}
  .num  {{ width:50px; text-align:center; color:#888;
           white-space:nowrap; }}
  .text {{ max-width:400px; line-height:1.55; }}
  .simp {{ color:#1a5276; }}
  .final{{ color:#145a32; font-weight:500; }}
  tr:hover td {{ background:#fff9c4 !important; transition:.1s; }}
</style>
</head>
<body>
<h1>Medical Discharge Summary → Indian Lay English (v2)</h1>
<p>{len(df)} summaries &mdash; open in <b>Chrome / Edge / Firefox</b> for best readability</p>
<table>
<thead><tr>
  <th>#</th>
  <th>Original Summary</th>
  <th>Rule-Based Simplified</th>
  <th style='background:#145a32'>Final Indian Lay English</th>
  <th>Flesch</th>
  <th>Ratio</th>
</tr></thead>
<tbody>{''.join(rows_html)}
</tbody>
</table>
</body></html>"""

with open(HTML_PATH, "w", encoding="utf-8") as _f:
    _f.write(html)
print(f"  OK  HTML report       → {HTML_PATH}")
print(f"      Download the .html file and open it in a browser for easy reading.")

## Step 11 — Summary Report

Aggregate statistics across all 200 processed summaries: readability improvement,
entity preservation audit, model vs rule-based usage, and top medical entities.

In [ ]:
sep = "=" * 65

print(sep)
print("             PIPELINE v2 — SUMMARY REPORT")
print(sep)
print(f"  Total summaries processed        : {len(df)}")
print(f"  Fine-tuning samples used         : {len(ft)}")
print()

# ── Readability ───────────────────────────────────────────────────
print("  --- Readability ---")
print(f"  Avg Flesch (original)            : {df['flesch_original'].mean():.2f}")
print(f"  Avg Flesch (rule-based)          : {df['flesch_rule_based'].mean():.2f}")
print(f"  Avg Flesch (final output)        : {df['flesch_simplified'].mean():.2f}")
print(f"  Avg Flesch improvement           : {df['flesch_improvement'].mean():.2f}")
improved_count = (df["flesch_improvement"] > 0).sum()
print(f"  Summaries with improvement       : {improved_count}/{len(df)}")
print()

# ── Compression ───────────────────────────────────────────────────
print("  --- Compression ---")
print(f"  Avg word count (original)        : {df['word_count_original'].mean():.1f}")
print(f"  Avg word count (simplified)      : {df['word_count_simplified'].mean():.1f}")
print(f"  Avg compression ratio            : {df['compression_ratio'].mean():.3f}")
print()

# ── Model vs Rule-based ──────────────────────────────────────────
total_chunks = model_used_count + rule_fallback_count
print("  --- Generation Strategy ---")
print(f"  Total chunks processed           : {total_chunks}")
print(f"  Model-generated (passed QC)      : {model_used_count} "
      f"({100*model_used_count/max(total_chunks,1):.1f}%)")
print(f"  Rule-based fallback              : {rule_fallback_count} "
      f"({100*rule_fallback_count/max(total_chunks,1):.1f}%)")
print()

# ── Entity audit ──────────────────────────────────────────────────
all_entities = []
for ent_str in df["medical_entities"]:
    try:
        ent_list = eval(ent_str) if isinstance(ent_str, str) else ent_str
        if isinstance(ent_list, list):
            for e in ent_list:
                if isinstance(e, dict) and "entity" in e:
                    all_entities.append(e["entity"].lower())
    except Exception:
        pass

entity_counts = Counter(all_entities)
print("  --- Top 15 Medical Entities ---")
for term, count in entity_counts.most_common(15):
    # Check if it appears in the output
    in_out = df["indian_lay_english"].str.contains(
        re.escape(term), case=False, na=False
    ).sum()
    print(f"    {term:<35s}  extracted={count:4d}  in_output={in_out:3d}/200")

print(f"\n{sep}")
print("  Pipeline v2 finished successfully.")
print(sep)

## Step 12 — Regional Indian Language Translation (IndicTrans2)

Translate the **Indian Lay English** output into **6 major regional Indian languages**
using [AI4Bharat IndicTrans2](https://github.com/AI4Bharat/IndicTrans2) (distilled 200M).

| Target Language | Code | Script |
|---|---|---|
| Hindi | `hin_Deva` | Devanagari |
| Telugu | `tel_Telu` | Telugu |
| Tamil | `tam_Taml` | Tamil |
| Bengali | `ben_Beng` | Bengali |
| Marathi | `mar_Deva` | Devanagari |
| Kannada | `kan_Knda` | Kannada |

**Strategy:**
1. Free the IndicBART model from GPU to reclaim VRAM
2. Load IndicTrans2-en-indic-dist-200M (lightweight, ~200M params)
3. Translate sentence-by-sentence (summaries avg 700+ words — too long for single pass)
4. Preserve numeric values and proper nouns where possible
5. Batch sentences for efficient GPU utilisation

In [ ]:
import gc
from IndicTransToolkit import IndicProcessor

# ── 12a.  Free IndicBART from GPU ────────────────────────────────
for obj_name in ["model", "trainer", "tokenizer", "data_collator"]:
    if obj_name in dir():
        try:
            del globals()[obj_name]
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"  GPU memory after cleanup: "
          f"{torch.cuda.memory_allocated()/1e9:.2f} GB allocated, "
          f"{torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

# ── 12b.  Load IndicTrans2 (en → Indic, distilled 200M) ─────────
INDICTRANS_MODEL = "ai4bharat/indictrans2-en-indic-dist-200M"

print(f"Loading {INDICTRANS_MODEL} ...")
it2_tokenizer = AutoTokenizer.from_pretrained(
    INDICTRANS_MODEL, trust_remote_code=True
)
it2_model = AutoModelForSeq2SeqLM.from_pretrained(
    INDICTRANS_MODEL, trust_remote_code=True, low_cpu_mem_usage=True
).to(device)
it2_model.eval()

ip = IndicProcessor(inference=True)

print(f"  OK  IndicTrans2 loaded on {device}")
print(f"  GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

# ── 12c.  Define translation targets ─────────────────────────────
TARGET_LANGUAGES = {
    "hindi":    "hin_Deva",
    "telugu":   "tel_Telu",
    "tamil":    "tam_Taml",
    "bengali":  "ben_Beng",
    "marathi":  "mar_Deva",
    "kannada":  "kan_Knda",
}
SRC_LANG = "eng_Latn"
BATCH_SIZE = 16          # sentences per batch (GPU-friendly)
MAX_SEQ_LEN = 256        # IndicTrans2 max tokens

# ── 12d.  Sentence-level translation function ────────────────────
import re as _re

def split_sentences(text):
    """Split text into sentences, preserving structure."""
    sents = _re.split(r'(?<=[.!?])\s+', str(text).strip())
    return [s for s in sents if s.strip()]

def translate_text(text, tgt_lang, batch_size=BATCH_SIZE):
    """Translate a full English text → target Indian language,
    processing sentence-by-sentence in batches."""
    sentences = split_sentences(text)
    if not sentences:
        return ""

    translated_sentences = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]

        # IndicProcessor preprocessing
        preprocessed = ip.preprocess_batch(
            batch, src_lang=SRC_LANG, tgt_lang=tgt_lang
        )

        inputs = it2_tokenizer(
            preprocessed,
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            generated = it2_model.generate(
                **inputs,
                num_beams=5,
                num_return_sequences=1,
                max_length=MAX_SEQ_LEN,
            )

        decoded = it2_tokenizer.batch_decode(
            generated, skip_special_tokens=True
        )

        # IndicProcessor postprocessing (de-romanise, normalise)
        postprocessed = ip.postprocess_batch(decoded, lang=tgt_lang)
        translated_sentences.extend(postprocessed)

    return " ".join(translated_sentences)

# ── 12e.  Translate all summaries into each target language ───────
print(f"\nTranslating {len(df)} summaries into {len(TARGET_LANGUAGES)} languages ...\n")

for lang_name, lang_code in TARGET_LANGUAGES.items():
    col = f"lang_{lang_name}"
    translations = []

    print(f"  [{lang_name.upper()}] ({lang_code})", end=" ", flush=True)

    for idx in tqdm(range(len(df)), desc=lang_name, leave=False):
        eng_text = str(df.at[idx, "indian_lay_english"])
        try:
            translated = translate_text(eng_text, lang_code)
        except Exception as e:
            translated = f"[TRANSLATION_ERROR] {e}"
        translations.append(translated)

    df[col] = translations
    print(f"  ✓  {len(translations)} summaries translated")

    # Checkpoint after each language
    checkpoint_path = f"/kaggle/working/translation_checkpoint_{lang_name}.xlsx"
    df.to_excel(checkpoint_path, index=False, engine="openpyxl")

print(f"\n  OK  All translations complete. Columns added: {list(TARGET_LANGUAGES.keys())}")
print(f"  GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## Step 13 — Save Multilingual Output (Excel + HTML)

Saves the full multilingual pipeline output:

| File | Contents |
|---|---|
| `multilingual_pipeline_output.xlsx` | All columns — English Lay + 6 regional languages, formatted |
| `multilingual_pipeline_output.html` | Browser-friendly report with **language tabs** for each regional language |

In [ ]:
import openpyxl
from openpyxl.styles import PatternFill, Alignment, Font
from openpyxl.utils import get_column_letter

# ── 13a.  Define output columns ──────────────────────────────────
LANG_COLS = [f"lang_{name}" for name in TARGET_LANGUAGES]
MULTI_OUTPUT_COLUMNS = [
    "translated discharge_summary",
    "indian_lay_english",
] + LANG_COLS + [
    "flesch_original", "flesch_simplified", "flesch_improvement",
    "word_count_original", "word_count_simplified",
    "medical_entities",
]

existing_cols = [c for c in MULTI_OUTPUT_COLUMNS if c in df.columns]
MULTI_XLSX = "/kaggle/working/multilingual_pipeline_output.xlsx"
MULTI_HTML = "/kaggle/working/multilingual_pipeline_output.html"

# ── 13b.  Write formatted Excel ──────────────────────────────────
df[existing_cols].to_excel(MULTI_XLSX, index=False, engine="openpyxl")

wb = openpyxl.load_workbook(MULTI_XLSX)
ws = wb.active

HEADER_FILL  = PatternFill("solid", fgColor="1F4E79")
HEADER_FONT  = Font(bold=True, color="FFFFFF", size=11)
ROW_ODD      = PatternFill("solid", fgColor="EBF3FB")
ROW_EVN      = PatternFill("solid", fgColor="FFFFFF")
WRAP_TOP     = Alignment(wrap_text=True, vertical="top")
CENTER_MID   = Alignment(wrap_text=False, vertical="center", horizontal="center")

TEXT_COLS = {"translated discharge_summary", "indian_lay_english",
             "medical_entities"} | set(LANG_COLS)
NUM_COLS  = {"flesch_original", "flesch_simplified", "flesch_improvement",
             "word_count_original", "word_count_simplified"}

col_names = [ws.cell(1, c).value for c in range(1, ws.max_column + 1)]

for col_idx, col_name in enumerate(col_names, start=1):
    letter = get_column_letter(col_idx)
    hdr = ws.cell(1, col_idx)
    hdr.fill = HEADER_FILL
    hdr.font = HEADER_FONT
    hdr.alignment = CENTER_MID
    if col_name in TEXT_COLS:
        ws.column_dimensions[letter].width = 60
    elif col_name in NUM_COLS:
        ws.column_dimensions[letter].width = 16
    else:
        ws.column_dimensions[letter].width = 22

for row_idx in range(2, ws.max_row + 1):
    fill = ROW_ODD if row_idx % 2 == 0 else ROW_EVN
    for col_idx, col_name in enumerate(col_names, start=1):
        cell = ws.cell(row_idx, col_idx)
        cell.fill = fill
        cell.alignment = WRAP_TOP if col_name in TEXT_COLS else Alignment(vertical="top")
    ws.row_dimensions[row_idx].height = 130

ws.freeze_panes = "A2"
wb.save(MULTI_XLSX)
print(f"  OK  Multilingual Excel → {MULTI_XLSX}")

# ── 13c.  Generate HTML report with language tabs ─────────────────
def _esc(s):
    return str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

# Build per-language tab content
lang_display = {
    "indian_lay_english": "English (Lay)",
    **{f"lang_{k}": k.title() for k in TARGET_LANGUAGES},
}

tab_buttons = []
tab_bodies  = []

for tab_idx, (col_key, display_name) in enumerate(lang_display.items()):
    active_cls = "active" if tab_idx == 0 else ""
    tab_buttons.append(
        f'<button class="tab-btn {active_cls}" '
        f'onclick="showTab(\'{col_key}\')" id="btn_{col_key}">'
        f'{display_name}</button>'
    )

    rows_html = []
    for i in range(len(df)):
        orig  = _esc(df.at[i, "translated discharge_summary"])
        trans = _esc(df.at[i, col_key]) if col_key in df.columns else "[N/A]"
        bg    = "#f0f7ff" if i % 2 == 0 else "#ffffff"
        rows_html.append(f"""
        <tr style='background:{bg}'>
          <td class='num'>{i+1}</td>
          <td class='text'>{orig}</td>
          <td class='text translated'>{trans}</td>
        </tr>""")

    display = "block" if tab_idx == 0 else "none"
    tab_bodies.append(f"""
    <div class="tab-content" id="tab_{col_key}" style="display:{display}">
      <table>
      <thead><tr>
        <th>#</th>
        <th>Original Discharge Summary</th>
        <th style='background:#145a32'>{display_name} Translation</th>
      </tr></thead>
      <tbody>{''.join(rows_html)}</tbody>
      </table>
    </div>""")

html = f"""<!DOCTYPE html>
<html lang='en'>
<head>
<meta charset='UTF-8'>
<title>Multilingual Medical Pipeline — Results</title>
<style>
  body  {{ font-family:'Segoe UI',Arial,sans-serif; font-size:13px;
           margin:0; padding:16px; background:#f0f4f8; }}
  h1    {{ color:#1F4E79; margin-bottom:4px; }}
  p     {{ color:#555; margin-top:0; }}
  .tabs {{ display:flex; gap:4px; margin:16px 0 0 0; flex-wrap:wrap; }}
  .tab-btn {{ padding:8px 18px; border:none; border-radius:6px 6px 0 0;
              cursor:pointer; font-size:13px; font-weight:600;
              background:#d6e4f0; color:#1F4E79; transition:.15s; }}
  .tab-btn.active {{ background:#1F4E79; color:#fff; }}
  .tab-btn:hover  {{ background:#a8c4df; }}
  table {{ border-collapse:collapse; width:100%; background:#fff;
           box-shadow:0 1px 6px rgba(0,0,0,.12); border-radius:0 6px 6px 6px;
           overflow:hidden; }}
  th    {{ background:#1F4E79; color:#fff; padding:10px 14px;
           text-align:left; position:sticky; top:0; z-index:2;
           white-space:nowrap; }}
  td    {{ padding:10px 14px; vertical-align:top;
           border-bottom:1px solid #dde6f0; }}
  .num  {{ width:50px; text-align:center; color:#888; white-space:nowrap; }}
  .text {{ max-width:500px; line-height:1.55; }}
  .translated {{ color:#145a32; font-weight:500; }}
  tr:hover td {{ background:#fff9c4 !important; transition:.1s; }}
</style>
<script>
function showTab(langKey) {{
  document.querySelectorAll('.tab-content').forEach(el => el.style.display='none');
  document.querySelectorAll('.tab-btn').forEach(el => el.classList.remove('active'));
  document.getElementById('tab_'+langKey).style.display='block';
  document.getElementById('btn_'+langKey).classList.add('active');
}}
</script>
</head>
<body>
<h1>Medical Discharge Summary — Multilingual Translation</h1>
<p>{len(df)} summaries &times; {len(TARGET_LANGUAGES)+1} languages
   (English Lay + {', '.join(k.title() for k in TARGET_LANGUAGES)})</p>
<div class="tabs">{''.join(tab_buttons)}</div>
{''.join(tab_bodies)}
</body></html>"""

with open(MULTI_HTML, "w", encoding="utf-8") as fh:
    fh.write(html)
print(f"  OK  Multilingual HTML → {MULTI_HTML}")
print(f"      Download and open in a browser for tabbed language view.")

## Step 14 — Multilingual Pipeline Summary Report

Final statistics: translation coverage, character/word counts per language,
sample translations, and error audit.

In [ ]:
sep = "=" * 65

print(sep)
print("        MULTILINGUAL PIPELINE — SUMMARY REPORT")
print(sep)
print(f"  Total summaries                  : {len(df)}")
print(f"  Source column                    : indian_lay_english")
print(f"  Translation model                : {INDICTRANS_MODEL}")
print(f"  Target languages                 : {len(TARGET_LANGUAGES)}")
print()

# ── Per-language statistics ────────────────────────────────────────
print("  --- Per-Language Statistics ---")
print(f"  {'Language':<12s}  {'Avg Chars':>10s}  {'Avg Words':>10s}  {'Errors':>7s}  {'Sample (first 80 chars)'}")
print(f"  {'-'*12}  {'-'*10}  {'-'*10}  {'-'*7}  {'-'*40}")

for lang_name in TARGET_LANGUAGES:
    col = f"lang_{lang_name}"
    if col not in df.columns:
        print(f"  {lang_name:<12s}  {'[MISSING]':>10s}")
        continue

    texts = df[col].astype(str)
    errors = texts.str.startswith("[TRANSLATION_ERROR]").sum()
    valid = texts[~texts.str.startswith("[TRANSLATION_ERROR]")]

    avg_chars = valid.str.len().mean()
    avg_words = valid.str.split().str.len().mean()
    sample = texts.iloc[0][:80].replace("\n", " ")

    print(f"  {lang_name:<12s}  {avg_chars:>10.0f}  {avg_words:>10.1f}  "
          f"{errors:>7d}  {sample}")

print()

# ── English baseline for comparison ───────────────────────────────
eng_texts = df["indian_lay_english"].astype(str)
print(f"  English Lay English baseline:")
print(f"    Avg chars  : {eng_texts.str.len().mean():.0f}")
print(f"    Avg words  : {eng_texts.str.split().str.len().mean():.1f}")
print()

# ── Sample translations (Row 0) ──────────────────────────────────
print("  --- Sample Translations (Summary #1, first 200 chars) ---")
print(f"  ENGLISH : {eng_texts.iloc[0][:200]}...")
for lang_name in TARGET_LANGUAGES:
    col = f"lang_{lang_name}"
    if col in df.columns:
        sample = df.at[0, col]
        print(f"  {lang_name.upper():<8s}: {str(sample)[:200]}...")

print(f"\n  Output files:")
print(f"    {MULTI_XLSX}")
print(f"    {MULTI_HTML}")
print(f"\n{sep}")
print("  Full multilingual pipeline finished successfully.")
print(sep)